# 01 — Baseline: Logistic Regression on raw 64×64 pixels

The dumbest possible classifier. Sets the floor for the comparison table.
Reads the same arrays produced by `00_data_setup.ipynb`.

In [6]:
# === Colab preamble: clone repo, mount Drive, set up paths ===
import os, sys, subprocess

REPO_URL  = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
REPO_PATH = "/content/melanoma-detection-ham10000"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Mount Drive (silently re-uses existing mount on re-run)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import config
config.ensure_drive_dirs()
print("Drive root :", config.DRIVE_ROOT)
print("Data dir   :", config.DATA_DIR)
print("Results dir:", config.RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root : /content/drive/MyDrive/melanoma
Data dir   : /content/drive/MyDrive/melanoma/data
Results dir: /content/drive/MyDrive/melanoma/results


In [7]:
import random, numpy as np, torch
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [8]:
# --- Load arrays ---
import numpy as np, cv2
from src.data import load_arrays_balanced
X, y, ids, idx_train, idx_val, idx_test = load_arrays_balanced(config.DATA_DIR)
print("BALANCED test set -> n:", len(idx_test),
      " mel/non-mel:", int(y[idx_test].sum()), "/", int((y[idx_test]==0).sum()))
print("X:", X.shape, "  test size:", len(idx_test))

X: (10015, 224, 224, 3)   test size: 1503


In [9]:
# --- Resize 224 -> 64 and flatten ---
def to_baseline_vec(X224):
    out = np.empty((len(X224), config.BASELINE_IMG_SIZE * config.BASELINE_IMG_SIZE * 3),
                   dtype=np.float32)
    for i, im in enumerate(X224):
        small = cv2.resize(im, (config.BASELINE_IMG_SIZE, config.BASELINE_IMG_SIZE),
                           interpolation=cv2.INTER_AREA)
        out[i] = small.reshape(-1).astype(np.float32) / 255.0
    return out

X_train = to_baseline_vec(X[idx_train])
X_val   = to_baseline_vec(X[idx_val])
X_test  = to_baseline_vec(X[idx_test])
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
print("X_train:", X_train.shape)

X_train: (7009, 12288)


In [10]:
# --- Train Logistic Regression with class_weight='balanced' ---
import time
from sklearn.linear_model import LogisticRegression
hp = dict(class_weight="balanced", max_iter=1000, random_state=config.SEED, solver="lbfgs")
clf = LogisticRegression(**hp)
t0 = time.time(); clf.fit(X_train, y_train); train_time = time.time() - t0
print(f"Trained in {train_time:.1f}s")

Trained in 214.1s


In [11]:
# --- Evaluate on test set ---
from src.evaluation import save_standard_outputs, time_inference

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

inf_ms = time_inference(lambda x: clf.predict(x), X_test[:200])

metrics = save_standard_outputs(
    method_name="baseline_logistic",
    results_dir=config.RESULTS_DIR,
    y_true=y_test, y_pred=y_pred, y_prob=y_prob,
    ids=ids[idx_test],
    hyperparameters=hp,
    train_time_sec=train_time,
    inference_time_per_image_ms=inf_ms,
)
{k: v for k, v in metrics.items() if k != "hyperparameters"}

{'accuracy': 0.7957418496340652,
 'precision': 0.25,
 'recall': 0.41916167664670656,
 'f1': 0.3131991051454139,
 'roc_auc': 0.7337480727168417,
 'train_time_sec': 214.13794445991516,
 'inference_time_per_image_ms': 0.0330052149996618}